# Building Resilient Agents with Kalibr

**Partner cookbook — [Kalibr Systems](https://kalibr.systems)**

---

## The Problem with Single-Model Agents

Most OpenAI-powered agents are written like this:

```python
from openai import OpenAI
client = OpenAI()
response = client.chat.completions.create(model="gpt-4o", messages=[...])
```

This works in demos. In production:
- Rate limits fire at unpredictable times
- Provider latency spikes during peak load
- Partial outages affect specific model versions
- Your agent silently degrades or errors out

**What you need:** route away from degraded paths automatically. This notebook shows how with [Kalibr](https://kalibr.systems).

---

## What is Kalibr?

Kalibr captures the outcome of every LLM call (success, failure, quality score 0–1), learns which model+parameter combinations work best for each goal, and routes the next call to the optimal path.

**Not:** observability (Langfuse/LangSmith), model gateway (LiteLLM/OpenRouter), or a proxy.

**Classification:** execution path routing based on outcome signals.

In [ ]:
!pip install kalibr openai anthropic

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "your-openai-api-key"
os.environ["ANTHROPIC_API_KEY"] = "your-anthropic-api-key"
os.environ["KALIBR_API_KEY"] = "your-kalibr-api-key"
os.environ["KALIBR_TENANT_ID"] = "your-kalibr-tenant-id"

## Part 1: The Fragile Agent (Before)

Standard customer support agent hardcoded to GPT-4o.

In [ ]:
from openai import OpenAI

client = OpenAI()

def support_agent_fragile(user_query: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful customer support agent."},
            {"role": "user", "content": user_query}
        ],
        max_tokens=300,
    )
    return response.choices[0].message.content

result = support_agent_fragile("How do I reset my password?")
print(result)

## Part 2: The Resilient Agent (After)

Two changes:
1. `import kalibr` — **must be the first import, before OpenAI**
2. Use `Router` instead of bare `OpenAI` client

The response interface is identical: `response.choices[0].message.content`

In [ ]:
# Restart kernel if openai was already imported above.
# kalibr MUST be the first import.
import kalibr  # noqa — must be first

from kalibr import Router

support_router = Router(
    goal="customer_support_response",
    paths=["gpt-4o", "claude-sonnet-4-20250514"],
    success_when=lambda output: output is not None and len(output) > 30,
)

def support_agent_resilient(user_query: str) -> str:
    response = support_router.completion(
        messages=[
            {"role": "system", "content": "You are a helpful customer support agent."},
            {"role": "user", "content": user_query}
        ],
        max_tokens=300,
    )
    return response.choices[0].message.content

result = support_agent_resilient("How do I reset my password?")
print(result)

## Part 3: Outcome Reporting

For tasks where structural validation isn't enough, report outcomes explicitly.

In [ ]:
from kalibr import Router

quality_router = Router(
    goal="customer_support_quality",
    paths=["gpt-4o", "claude-sonnet-4-20250514", "gpt-4o-mini"],
)

response = quality_router.completion(
    messages=[
        {"role": "system", "content": "You are a helpful customer support agent."},
        {"role": "user", "content": "How do I update my billing information?"}
    ],
    max_tokens=300,
)
output = response.choices[0].message.content

has_steps = any(w in output.lower() for w in ["click", "go to", "select", "open"])
quality_score = 0.9 if has_steps else 0.4

quality_router.report(success=quality_score >= 0.5, score=quality_score)
print(f"Quality score: {quality_score}\nResponse: {output}")

## Benchmark Results

From Kalibr's internal testing during simulated GPT-4o degradation (70% failure rate):

| Agent Setup | Task Success Rate |
|---|---|
| Hardcoded `gpt-4o` | 16–36% |
| Retry with backoff | 40–55% |
| Kalibr-routed (`gpt-4o` + `claude-sonnet-4-20250514`) | 88–100% |

## What Changed

| | Before | After |
|---|---|---|
| Import | `from openai import OpenAI` | `import kalibr` first |
| Client | `OpenAI()` | `Router(goal, paths, success_when)` |
| Completion | `client.chat.completions.create(model=..., ...)` | `router.completion(messages=...)` |
| Fallback | None | Automatic |
| Improvement | Static | Continuous |

## Links
- [Kalibr docs](https://kalibr.systems/docs)
- [GitHub](https://github.com/kalibr-ai/kalibr-sdk-python)
- [Dashboard](https://dashboard.kalibr.systems)
- [PyPI](https://pypi.org/project/kalibr/)